# PyLauncher Parameter Sweeps with dapi

This notebook demonstrates how to use dapi's parameter sweep utilities to generate
PyLauncher task lists and submit sweep jobs on DesignSafe.

**PyLauncher** runs many independent serial tasks within a single SLURM allocation —
ideal for parameter studies, Monte Carlo simulations, and batch processing.

**What this notebook covers:**

1. **Generic demo** — a minimal `simulate.py` with `--alpha`/`--beta` parameters
2. **OpenSees demo** — Silvia Mazzoni's cantilever pushover with `--NodalMass`/`--LCol` sweep

In [ ]:
%pip install --user dapi --quiet

In [ ]:
import os
from dapi import DSClient

ds = DSClient()

---

## Part 1: Generic Demo

A simple example sweeping over two parameters (`--alpha`, `--beta`). The script
computes `result = alpha * beta`, writes it to a JSON output file, and prints a summary.
This pattern works with any app — the commands in `runsList.txt` are just shell commands.

### Write the script

In [ ]:
input_dir_generic = os.path.expanduser("~/MyData/pylauncher_demo/")
os.makedirs(input_dir_generic, exist_ok=True)

simulate_script = '''\
"""simulate.py — minimal demo script for PyLauncher parameter sweeps.

Accepts --alpha, --beta, and --output via command line.
Computes result = alpha * beta and writes it to the output directory.
"""
import argparse
import os
import json

parser = argparse.ArgumentParser()
parser.add_argument("--alpha", type=float, required=True)
parser.add_argument("--beta", type=float, required=True)
parser.add_argument("--output", type=str, required=True)
args = parser.parse_args()

os.makedirs(args.output, exist_ok=True)

result = args.alpha * args.beta
summary = {"alpha": args.alpha, "beta": args.beta, "result": result}

outfile = os.path.join(args.output, "result.json")
with open(outfile, "w") as f:
    json.dump(summary, f, indent=2)

print(f"alpha={args.alpha}, beta={args.beta} -> result={result:.4f} written to {outfile}")
'''

with open(os.path.join(input_dir_generic, "simulate.py"), "w") as f:
    f.write(simulate_script)

print(f"Wrote {input_dir_generic}simulate.py")

### Define the sweep

In [ ]:
sweep = {
    "ALPHA": [0.3, 0.5, 3.7],
    "BETA":  [1.1, 2.0, 3.0],
}

### Preview (dry run)

In [ ]:
ds.jobs.parametric_sweep.generate(
    'python3 simulate.py --alpha ALPHA --beta BETA --output out_ALPHA_BETA',
    sweep,
    preview=True,
)

### Generate sweep files

In [ ]:
commands = ds.jobs.parametric_sweep.generate(
    'python3 simulate.py --alpha ALPHA --beta BETA --output out_ALPHA_BETA',
    sweep,
    input_dir_generic,
    debug="host+job",
)

print(f"Generated {len(commands)} task commands\n")
print("=== runsList.txt ===")
with open(os.path.join(input_dir_generic, "runsList.txt")) as f:
    print(f.read())

print("=== call_pylauncher.py ===")
with open(os.path.join(input_dir_generic, "call_pylauncher.py")) as f:
    print(f.read())

print("=== Files in input directory ===")
for fn in sorted(os.listdir(input_dir_generic)):
    print(f"  {fn}")

### Submit

Replace `your_allocation` with your TACC allocation and uncomment to run.

In [ ]:
# job = ds.jobs.parametric_sweep.submit(
#     "/MyData/pylauncher_demo/",
#     app_id="agnostic",
#     allocation="your_allocation",
#     node_count=1,
#     cores_per_node=48,
#     max_minutes=30,
# )
# job.monitor()

---

## Part 2: OpenSees Cantilever Pushover Sweep

A real-world example based on Silvia Mazzoni's cantilever pushover analysis.
We sweep over `NodalMass` and `LCol` (column length) to study how these structural
parameters affect the pushover response.

The cantilever model:
```
   ^Y
   |
   2       __
   |         |
   |         |
   |         |
 (1)      LCol
   |         |
   |         |
   |         |
 =1=    ----  -------->X
```

- Node 1: fixed base
- Node 2: free top with `NodalMass`
- Elastic beam-column element
- Gravity load (2000 kip downward) followed by lateral pushover (displacement-controlled)

### Write the analysis script

This is the OpenSeesPy cantilever pushover script adapted from
[Silvia Mazzoni's example](https://opensees.berkeley.edu/wiki/index.php/Examples_Manual).
It accepts `--NodalMass`, `--LCol`, and `--outDir` as command-line arguments
so PyLauncher can run each parameter combination independently.

In [ ]:
input_dir_opensees = os.path.expanduser("~/MyData/opensees_sweep/")
os.makedirs(input_dir_opensees, exist_ok=True)

cantilever_script = '''\
# Ex1a.Canti2D.Push — OpenSeesPy cantilever pushover
# Adapted from Silvia Mazzoni & Frank McKenna, 2006/2020
# Units: kip, inch, second
#
# Command-line arguments (set by PyLauncher per task):
#   --NodalMass  mass at free node
#   --LCol       column length
#   --outDir     output directory for this run

import argparse
import os

if os.path.exists("opensees.so"):
    import opensees as ops
else:
    import openseespy.opensees as ops

parser = argparse.ArgumentParser()
parser.add_argument("--NodalMass", type=float, required=True)
parser.add_argument("--LCol", type=float, required=True)
parser.add_argument("--outDir", type=str, required=True)
args = parser.parse_args()

NodalMass = args.NodalMass
LCol = args.LCol
outDir = args.outDir

os.makedirs(outDir, exist_ok=True)
print(f"Running: NodalMass={NodalMass}, LCol={LCol}, outDir={outDir}")

ops.wipe()
ops.model("basic", "-ndm", 2, "-ndf", 3)

# Geometry
ops.node(1, 0, 0)
ops.node(2, 0, LCol)
ops.fix(1, 1, 1, 1)
ops.mass(2, NodalMass, 0.0, 0.0)

# Element
ops.geomTransf("Linear", 1)
ops.element("elasticBeamColumn", 1, 1, 2, 3600000000, 4227, 1080000, 1)

# Recorders
ops.recorder("Node", "-file", f"{outDir}/DFree.out", "-time", "-node", 2, "-dof", 1, 2, 3, "disp")
ops.recorder("Node", "-file", f"{outDir}/RBase.out", "-time", "-node", 1, "-dof", 1, 2, 3, "reaction")
ops.recorder("Element", "-file", f"{outDir}/FCol.out", "-time", "-ele", 1, "globalForce")

# Gravity analysis
ops.timeSeries("Linear", 1)
ops.pattern("Plain", 1, 1)
ops.load(2, 0.0, -2000.0, 0.0)
ops.wipeAnalysis()
ops.constraints("Plain")
ops.numberer("Plain")
ops.system("BandGeneral")
ops.test("NormDispIncr", 1.0e-8, 6)
ops.algorithm("Newton")
ops.integrator("LoadControl", 0.1)
ops.analysis("Static")
ops.analyze(10)
ops.loadConst("-time", 0.0)

# Pushover analysis
ops.timeSeries("Linear", 2)
ops.pattern("Plain", 2, 2)
ops.load(2, 2000.0, 0.0, 0.0)
ops.integrator("DisplacementControl", 2, 1, 0.1)
ops.analyze(1000)

print(f"Done: NodalMass={NodalMass}, LCol={LCol}")
'''

with open(os.path.join(input_dir_opensees, "cantilever.py"), "w") as f:
    f.write(cantilever_script)

print(f"Wrote {input_dir_opensees}cantilever.py")

### Define the sweep

In [ ]:
sweep_opensees = {
    "NODAL_MASS": [4.19, 4.39, 4.59, 4.79, 4.99],
    "LCOL": [100, 200, 300],
}

### Preview

In [ ]:
df = ds.jobs.parametric_sweep.generate(
    "python3 cantilever.py --NodalMass NODAL_MASS --LCol LCOL --outDir out_NODAL_MASS_LCOL",
    sweep_opensees,
    preview=True,
)
print(f"Total runs: {len(df)}")
df

### Generate sweep files

In [ ]:
commands = ds.jobs.parametric_sweep.generate(
    "python3 cantilever.py --NodalMass NODAL_MASS --LCol LCOL --outDir out_NODAL_MASS_LCOL",
    sweep_opensees,
    input_dir_opensees,
)

print(f"Generated {len(commands)} task commands\n")
print("=== runsList.txt ===")
with open(os.path.join(input_dir_opensees, "runsList.txt")) as f:
    print(f.read())

print("=== call_pylauncher.py ===")
with open(os.path.join(input_dir_opensees, "call_pylauncher.py")) as f:
    print(f.read())

print("=== Files in input directory ===")
for fn in sorted(os.listdir(input_dir_opensees)):
    print(f"  {fn}")

### Submit

Replace `your_allocation` with your TACC allocation and uncomment to run.

In [ ]:
# job = ds.jobs.parametric_sweep.submit(
#     "/MyData/opensees_sweep/",
#     app_id="openseespy-s3",
#     allocation="your_allocation",
#     node_count=1,
#     cores_per_node=48,
#     max_minutes=30,
# )
# job.monitor()